In [30]:
import pandas as pd
import numpy as np

In [31]:
print("Starting the Airbnb data cleaning pipeline...\n")
# Using low_memory=False to avoid warnings with large files
df = pd.read_csv('listings.csv', low_memory=False, encoding='utf-8')

Starting the Airbnb data cleaning pipeline...



In [32]:

# 1. FINANCIAL CLEANING (Prices)

print("1. Handling financial formatting...")
if df['price'].dtype == 'O': 
    df['price'] = df['price'].astype(str).str.replace('$', '', regex=False)
    df['price'] = df['price'].str.replace(',', '', regex=False)
    df['price'] = pd.to_numeric(df['price'], errors='coerce')

1. Handling financial formatting...


In [33]:
# 2. FEATURE ENGINEERING: TEXT EXTRACTION (Regex)

print("2. Extracting hidden numbers in text (Bathrooms)...")
if 'bathrooms_text' in df.columns:
    # Goes to the string "1.5 shared baths" and extracts only the "1.5"
    df['bathrooms_clean'] = df['bathrooms_text'].astype(str).str.extract(r'(\d+\.?\d*)')[0].astype(float)
    df['bathrooms_clean'] = df['bathrooms_clean'].fillna(1.0) # Assumes 1 if it is empty
else:
    df['bathrooms_clean'] = 1.0

2. Extracting hidden numbers in text (Bathrooms)...


In [34]:
# 3. TRUST CONVERSION (Booleans to 1/0)

print("3. Mapping business rules and trust...")
booleans_cols = ['host_is_superhost', 'instant_bookable', 'host_identity_verified', 'host_has_profile_pic']
for col in booleans_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).map({'t': 1, 'f': 0, 'True': 1, 'False': 0})
        df[col] = df[col].fillna(0)

3. Mapping business rules and trust...


In [35]:
# 4. MISSING VALUES HANDLING & FEATURE ENGINEERING

print("4. Calculating Value for Money and filling nulls...")
df['number_of_reviews'] = df['number_of_reviews'].fillna(0)
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

if 'review_scores_rating' in df.columns:
    df['review_scores_rating'] = df['review_scores_rating'].fillna(df['review_scores_rating'].median())

# Price per Guest (Value for money)
if 'accommodates' in df.columns:
    df['price_per_person'] = np.where(df['accommodates'] > 0, df['price'] / df['accommodates'], 0)

# Fill empty bedrooms and beds with the median
for col in ['bedrooms', 'beds', 'accommodates']:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

4. Calculating Value for Money and filling nulls...


In [ ]:
# 5. FINAL FILTERING FOR THE MODEL

print("5. Compiling the final table for Artificial Intelligence...")
target_columns = [
    'id', 'latitude', 'longitude', 'neighbourhood_cleansed', # We added these two!
    'room_type', 'accommodates', 'bedrooms', 'beds', 'bathrooms_clean',
    'price', 'price_per_person', 'minimum_nights', 'availability_365',
    'host_is_superhost', 'host_identity_verified', 'host_has_profile_pic', 
    'instant_bookable', 'number_of_reviews', 'reviews_per_month', 'review_scores_rating'
]

# Ensure that only the columns that exist in the dataset are selected
final_columns = [col for col in target_columns if col in df.columns]
df_clean = df[final_columns].dropna(subset=['price'])

# Remove absurd prices/insertion errors that mess up statistics (> 1000€/night)
df_clean = df_clean[df_clean['price'] < 1000]

# Save the clean dataset 
df_clean.to_csv('airbnb_ready_for_ml.csv', index=False)

print(f"\n✅ SUCCESS! Bulletproof database with {len(df_clean)} active listings.")
print("File 'airbnb_ready_for_ml.csv' generated.")

5. Compiling the final table for Artificial Intelligence...

✅ SUCCESS! Bulletproof database with 21164 active listings.
File 'airbnb_ready_for_ml.csv' generated.
